In [1]:
import numpy as np
from scipy.optimize import fsolve, root, least_squares, brentq

![dissipator circuit](full_diss_circuit.svg)

# Constants

In [2]:
e = 1.602176634e-19   # elementary charge, C
h = 6.626e-34          # Planck constant, J*s
hbar = 1.05457e034  # reduced Planck constant, J*s

fF = 1e-15
nH = 1e-9

# 1. Dissipator-Qubit Coupling

![dissipator circuit](qubit_dissipator_coupling.svg)

## A. Define Functions (Equations)

$$ E_C^Q = \frac{e^2 C_D+C_J}{2 C_D [ C_Q +C_J ]+C_QC_J}$$


$$E_C^D = \frac{e^2 C_Q +C_J}{2 C_D [ C_Q +C_J ]+C_QC_J}$$

In [3]:
def ECQ(CJ, CQ, CD):
    return (e**2 / 2) * ((CD + CJ) / (CD * CQ + CD * CJ + CQ * CJ))


def ECD(CJ, CQ, CD):
    return (e**2 / 2) * ((CQ + CJ) / (CD * CQ + CD * CJ + CQ * CJ))


and with a desired $\frac{E_J}{E_C}$ ratio, we can get $E_J$ by itself

In [4]:
def EJD(ratioD, CJ, CQ, CD):
    return ratioD * ECD(CJ, CQ, CD)


def EJQ(ratioQ, CJ, CQ, CD):
    return ratioQ * ECQ(CJ, CQ, CD)

with denoted transmon frequencies as: 
$$\omega_Q  \approx \sqrt{8E_J^QE_C^Q}-E_C^Q$$
$$\omega_D \approx \sqrt{8E_J^DE_C^D}-E_C^D$$

In [5]:
def omegaQ(ratioQ,CJ, CQ, CD):
    return np.sqrt(8 * ECQ(CJ, CQ, CD) * EJQ(ratioQ, CJ, CQ, CD)) - ECQ(CJ, CQ, CD)


def omegaD(ratioD,CJ, CQ, CD):
    return np.sqrt(8 * ECD(CJ, CQ, CD) * EJD(ratioD,CJ, CQ, CD)) - ECD(CJ, CQ, CD)

where anharmonicities are defined as:
$$\eta_Q \approx - E_C^Q$$
$$\eta_D \approx - E_C^D$$

coupling $J$ is then,
$$ J = -4e^2 \frac{C_J}{C_J C_Q + C_J C_D+C_D C_D}\left( \frac{E_J^Q}{32 E_C^Q} \frac{E_J^D}{32E_C^D}\right)^{1/4} $$

then we now need to be able to solve for a $C_J$ that gives us the desired $J$.

We typically have the $E_J/E_C$ ratios in mind beforehand, as well as the desired qubit/dissipator frequencies.

In [6]:
def J(ratioQ, ratioD, CJ, CQ, CD):
    return (
       ( -4
        * e**2)
        * (CJ / (CD * CQ + CD * CJ + CQ * CJ))
        * ( EJQ(ratioQ, CJ, CQ, CD) * EJD(ratioD, CJ, CQ, CD)/ (32*ECQ(CJ, CQ, CD)*32*ECD(CJ, CQ, CD))) ** (1 / 4)
    )


## B. Solver

### Target $E_J/E_C$ ratios

sometimes your $E_J/E_C$ ratio can be determined by your desired anharmonicity and qubit frequency

In [7]:
targetOmegaQ = 4.5e9 # Qubit frequency
targetOmegaD = 7.0e9 # Dissipator Frequency
targetEtaQ = -200e6 # anharmonicity


In [8]:
def solve_EJ_ratio(omega,eta):
    E_C = -eta
    E_J = (omega + E_C)**2 / 8/ E_C

    print(f'Target EJ: {E_J/1e9} GHz')
    print(f'Target EJ/EC: {E_J/E_C}')

    return E_J/E_C

In [9]:
ratioQ = solve_EJ_ratio(targetOmegaQ,targetEtaQ)

Target EJ: 13.80625 GHz
Target EJ/EC: 69.03125


and other times, we have a desired $E_J/E_C$ ratio at a specified frequency,

In [10]:
ratioD = 100 # from given parameters

In [11]:
def solve_EC_from_ratio(omega,ratio):
    E_C = omega / (np.sqrt(8*ratio)-1)

    print(f'Target EC: {E_C/1e6} MHz')
    return E_C

In [12]:
solve_EC_from_ratio(4.5e9,70)

Target EC: 198.54997512664238 MHz


np.float64(198549975.12664238)

In [13]:
EC_D = solve_EC_from_ratio(targetOmegaD,ratioD)

Target EC: 256.55807100404667 MHz


Convert from frequencies to energy units

In [14]:
targetOmegaQh = targetOmegaQ * h # Qubit frequency
targetOmegaDh = targetOmegaD * h # Dissipator Frequency
targetJh =  -200e6 * h # Target coupliing

In [15]:
def equations(x_fF,targetOmegaQ,targetOmegaD,targetJ,ratioQ,ratioD):

    CJ, CQ, CD = [v * 1e-15 for v in x_fF]
    return [
        (omegaQ(ratioQ,CJ, CQ, CD) - targetOmegaQ) / targetOmegaQ,
        (omegaD(ratioD,CJ, CQ, CD) - targetOmegaD) / targetOmegaD,
        (J(ratioQ, ratioD, CJ, CQ, CD)- targetJ) / targetJ,
    ]

Provide initial guesses in femtoFarads

we can figure out relative guesses for capacitances given frequencies, and $E_J$ or $E_C$
using this site:

[Click here to visit Transmon Qubit Calculator](https://antonpotocnik.com/?p=560257)

In [16]:
# qubit-diss capacitance, qubit capcaitance, dissipator capacitance inital guesses
x0_fF = [1.89, 97, 75]

Now we can solve:

In [17]:
sol_fF = fsolve(equations, x0_fF,args=(targetOmegaQh,targetOmegaDh,targetJh,ratioQ,ratioD), full_output=False)
CJsol, CQsol, CDsol = [v * 1e-15 for v in sol_fF]
print(f"CJ = {CJsol / 1e-15} fF")
print(f"CQ = {CQsol / 1e-15} fF")
print(f"CD = {CDsol / 1e-15} fF")


CJ = 5.88463374569198 fF
CQ = 91.42404058610025 fF
CD = 69.97240240286669 fF


## C. Verify

In [18]:
print(f"Verify omegaQ = {omegaQ(ratioQ,CJsol, CQsol, CDsol) / h / 1e9} GHz (target: {targetOmegaQ/1e9})")
print(f"Verify omegaD = {omegaD(ratioD,CJsol, CQsol, CDsol) / h / 1e9} GHz (target: {targetOmegaD/1e9})")
print(f"Verify J      = {J(ratioQ,ratioD,CJsol, CQsol, CDsol) / h / 1e6} MHz (target: {targetJh/h})")
print(f"Verify EJQ/ECQ = {EJQ(ratioQ,CJsol, CQsol, CDsol) / ECQ(CJsol, CQsol, CDsol)} (target: {ratioQ})")  # NOTE: the original notebook's Print statement says "target: 50" here even though ratioQ = 70 -- likely a leftover/typo in the original, kept as-is for fidelity
print(f"Verify EJD/ECD = {EJD(ratioD,CJsol, CQsol, CDsol) / ECD(CJsol, CQsol, CDsol)} (target: {ratioD})")



Verify omegaQ = 4.500000000000001 GHz (target: 4.5)
Verify omegaD = 6.999999999999998 GHz (target: 7.0)
Verify J      = -199.99999999999866 MHz (target: -200000000.0)
Verify EJQ/ECQ = 69.03125 (target: 69.03125)
Verify EJD/ECD = 100.0 (target: 100)


# 2. Qubit-Resonator Coupling

Now let's consider the case of a readout resonator capacitively coupled to a transmon qubit.

We can model the resonator as LC oscillator, and the circuit diagram looks like this:

![qubit-resonator circuit](qubit_resonator_coupling.svg)

The Hamiltonian for this circuit is given by:
$$    \hat{H} = \omega_q \hat{a}^\dagger\hat{a} +\frac{\eta_Q}{2} \hat{a}^\dagger\hat{a}(\hat{a}^\dagger\hat{a}-1) +\omega_r \hat{c}^\dagger \hat{c} +g(\hat{a}-\hat{a}^\dagger)(\hat{c}-\hat{c}^\dagger).$$



## A. Define Functions (Equations)

Where the resonator frequency is:
$$\omega_r = \sqrt{8 E_J \tilde{E}_C^R}$$
with
$$E_L = \frac{\hbar^2}{4e^2L}=\frac{\varphi_0}{L}$$

The transmon (qubit) frequency is, again,
$$\omega_Q = \sqrt{8E_J\tilde{E}_C^Q} -\tilde{E}_C^Q$$
with anharmonicity
$$\eta_Q \approx  -\tilde{E}_C^Q$$
and
$$ \tilde{E}_C^Q = \frac{e^2}{2}\frac{C_R +C_g}{\tilde{C}_Q[C_R+C_g]+C_RC_g}$$

we also can denote 
$$ E_C^R  = \frac{c^2}{2}\frac{\tilde{C}_Q +C_g}{C_R(\tilde{C}_R+C_g)+C_QC_g}$$

Finally, the coupling $g$ is given by:
$$g = -4e^2 \frac{C_g}{C_R\tilde{C}_Q+C_RC_g + \tilde{C}_QC_g} \left( \frac{E_L^R}{32E_C^R}\frac{E_J^Q}{32\tilde{E}_C^Q}\right)^{1/4} \left(1+\frac{\eta_Q}{2\omega_Q}\right)$$

## B. Solver

In [23]:
from capacitance_solver import solve, g_func

In [20]:
targetOmegaQ_Hz = 4.5e9
targetOmegaR_Hz = 5.75e9
targetECQ_Hz    = 200e6
targetg_Hz      = -50e6

x = solve(targetOmegaQ_Hz,targetOmegaR_Hz,targetECQ_Hz,targetg_Hz,False)


── Solution ─────────────────────────────
  CQ = 96.3885 fF
  CR = 6.1963 fF
  Cg = 0.5012 fF
  L  = 114.4376  nH

── Verification ─────────────────────────
  ECQ = 200.0000 MHz   (target: 200.0)
  fQ  = 4.5000 GHz   (target: 4.500)
  fR  = 5.7500 GHz   (target: 5.750)
  g   = -50.0000 MHz   (target: -50.0)

  max_residual = 1.97e-16  ✓ converged


In [22]:
targetOmegaQ_Hz = 4.5e9
targetOmegaR_Hz = 5.75e9
targetECQ_Hz    = 198e6
targetg_Hz      = -50e6

x = solve(targetOmegaQ_Hz,targetOmegaR_Hz,targetECQ_Hz,targetg_Hz,False)


── Solution ─────────────────────────────
  CQ = 97.2486 fF
  CR = 9.5200 fF
  Cg = 0.6197 fF
  L  = 75.5885  nH

── Verification ─────────────────────────
  ECQ = 198.0000 MHz   (target: 198.0)
  fQ  = 4.5000 GHz   (target: 4.500)
  fR  = 5.7500 GHz   (target: 5.750)
  g   = -50.0000 MHz   (target: -50.0)

  max_residual = 2.87e-16  ✓ converged


In [ ]:
g_func(69,70e-9)

# 3. Dissipator-Filter couplings

The functions and system are actually the same as the Qubit-resonator just at different parameter choices, so all we need to do is solve!

In [21]:
targetOmegaD_Hz = 4.5e9
targetOmegaF_Hz =9.5e9
targetECD_Hz    = solve_EC_from_ratio(targetOmegaF_Hz,ratioD)
targetg_Hz      = -50e6
x = solve(targetOmegaD_Hz,targetOmegaF_Hz,targetECD_Hz,targetg_Hz,False)

Target EC: 348.1859535054919 MHz

── Solution ─────────────────────────────
  CQ = 52.3169 fF
  CR = 844.1511 fF
  Cg = 3.3286 fF
  L  = 0.3313  nH

── Verification ─────────────────────────
  ECQ = 348.1860 MHz   (target: 348.2)
  fQ  = 4.5000 GHz   (target: 4.500)
  fR  = 9.5000 GHz   (target: 9.500)
  g   = -50.0000 MHz   (target: -50.0)

  max_residual = 0.00e+00  ✓ converged
